In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt

# Column names from dataset description
column_names = [
    "mpg", "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year", "origin", "car_name"
]

# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
df = pd.read_csv(url, names=column_names, sep=r'\s+', na_values="?")

# Drop non-numeric column
df_numeric = df.drop(columns=["car_name"])

# Drop rows with any missing values (clean dataset)
df_clean = df_numeric.dropna().reset_index(drop=True)

# Mask 50% of observed horsepower values
mask = df_clean.sample(frac=0.5, random_state=42).index
true_vals = df_clean.loc[mask, "horsepower"]

df_masked = df_clean.copy()
df_masked.loc[mask, "horsepower"] = np.nan

# Imputation Methods

# Linear Regression Imputation
train_lr = df_masked[df_masked["horsepower"].notnull()]
test_lr = df_masked[df_masked["horsepower"].isnull()]

X_train_lr = train_lr.drop(columns=["horsepower"])
y_train_lr = train_lr["horsepower"]
X_test_lr = test_lr.drop(columns=["horsepower"])

lin_reg = LinearRegression()
lin_reg.fit(X_train_lr, y_train_lr)

df_lr = df_masked.copy()
df_lr.loc[df_lr["horsepower"].isnull(), "horsepower"] = lin_reg.predict(X_test_lr)

rmse_lr = np.sqrt(mean_squared_error(true_vals, df_lr.loc[mask, "horsepower"]))

# KNN Imputation
knn_imputer = KNNImputer(n_neighbors=5)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_masked), columns=df_masked.columns)
rmse_knn = np.sqrt(mean_squared_error(true_vals, df_knn.loc[mask, "horsepower"]))

# MICE Imputation
mice_imputer = IterativeImputer(random_state=0)
df_mice = pd.DataFrame(mice_imputer.fit_transform(df_masked), columns=df_masked.columns)
rmse_mice = np.sqrt(mean_squared_error(true_vals, df_mice.loc[mask, "horsepower"]))

# Compare RMSE values
rmse_values = {
    'Linear Regression': rmse_lr,
    'KNN': rmse_knn,
    'MICE': rmse_mice
}

plt.figure(figsize=(8, 6))
sns.barplot(x=list(rmse_values.keys()), y=list(rmse_values.values()))
plt.ylabel("RMSE")
plt.title("RMSE Comparison of Imputation Methods")
plt.savefig("rmse_comparison_chart.png")
plt.close()

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original observed distribution
sns.kdeplot(df_clean["horsepower"], fill=True, ax=axes[0, 0], color="black")
axes[0, 0].set_title("Original Horsepower Distribution")

# Linear Regression Imputation
sns.kdeplot(df_lr["horsepower"], fill=True, ax=axes[0, 1], color="blue")
axes[0, 1].set_title(f"Linear Regression Imputation (RMSE={rmse_lr:.2f})")

# KNN Imputation
sns.kdeplot(df_knn["horsepower"], fill=True, ax=axes[1, 0], color="green")
axes[1, 0].set_title(f"KNN Imputation (RMSE={rmse_knn:.2f})")

# MICE Imputation
sns.kdeplot(df_mice["horsepower"], fill=True, ax=axes[1, 1], color="red")
axes[1, 1].set_title(f"MICE Imputation (RMSE={rmse_mice:.2f})")

for ax in axes.flat:
    ax.set_xlabel("Horsepower")
    ax.set_ylabel("Density")

plt.tight_layout()
plt.savefig("horsepower_imputation_kdes.png")
plt.close()

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress convergence warnings for SGD
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Load and prepare dataset
column_names = [
    "mpg", "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year", "origin", "car_name"
]
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
df = pd.read_csv(url, names=column_names, sep=r'\s+', na_values="?")

df_numeric = df.drop(columns=["car_name"])
df_clean = df_numeric.dropna().reset_index(drop=True)

# Mask 50% of horsepower values
mask = df_clean.sample(frac=0.5, random_state=42).index
df_masked = df_clean.copy()
df_masked.loc[mask, "horsepower"] = np.nan

# Train/test splits
train_data = df_masked[df_masked["horsepower"].notnull()]
test_data = df_masked[df_masked["horsepower"].isnull()]
true_vals_indices = test_data.index
true_vals = df_clean.loc[true_vals_indices, "horsepower"].values

feature_names = train_data.drop(columns=["horsepower"]).columns
X_train = train_data.drop(columns=["horsepower"]).values
y_train = train_data["horsepower"].values
X_test = test_data.drop(columns=["horsepower"]).values

# Linear Regression (OLS)
# WITHOUT NORMALIZATION
lr_no_norm = LinearRegression()
lr_no_norm.fit(X_train, y_train)
y_pred_no_norm = lr_no_norm.predict(X_test)
rmse_no_norm = np.sqrt(mean_squared_error(true_vals, y_pred_no_norm))
r2_no_norm = r2_score(true_vals, y_pred_no_norm)

# Standard normalization
scaler_standard = StandardScaler()
X_train_std = scaler_standard.fit_transform(X_train)
X_test_std = scaler_standard.transform(X_test)

lr_std = LinearRegression()
lr_std.fit(X_train_std, y_train)
y_pred_std = lr_std.predict(X_test_std)
rmse_std = np.sqrt(mean_squared_error(true_vals, y_pred_std))
r2_std = r2_score(true_vals, y_pred_std)

# -----------------------------
# SGD Regressor
# -----------------------------
# SGD without normalization
sgd_no_norm = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd_no_norm.fit(X_train, y_train)
y_pred_sgd_no_norm = sgd_no_norm.predict(X_test)
rmse_sgd_no_norm = np.sqrt(mean_squared_error(true_vals, y_pred_sgd_no_norm))
r2_sgd_no_norm = r2_score(true_vals, y_pred_sgd_no_norm)

# SGD with standard normalization
sgd_std = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd_std.fit(X_train_std, y_train)
y_pred_sgd_std = sgd_std.predict(X_test_std)
rmse_sgd_std = np.sqrt(mean_squared_error(true_vals, y_pred_sgd_std))
r2_sgd_std = r2_score(true_vals, y_pred_sgd_std)


# Visualization: Feature Scales
plt.style.use('seaborn-v0_8-talk')
# Updated to 1x2 plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Impact of Normalization on Feature Scales', fontsize=20, y=1.03)

sns.boxplot(data=pd.DataFrame(X_train, columns=feature_names), ax=axes[0])
axes[0].set_title('Original Features')
axes[0].set_ylabel('Value')

sns.boxplot(data=pd.DataFrame(X_train_std, columns=feature_names), ax=axes[1])
axes[1].set_title('Standard Normalized (Z-score)')
axes[1].set_ylabel('Standardized Value')
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('feature_scale_comparison.png', dpi=100, bbox_inches='tight')
plt.close()

# Visualization 2: Model Performance
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Impact of Normalization on Model Performance', fontsize=20, y=1.05)

# Updated methods list
methods = ['No Norm', 'Standard Norm']
# Updated RMSE lists
rmse_lr_values = [rmse_no_norm, rmse_std]
rmse_sgd_values = [np.nan, rmse_sgd_std] # Keep nan for failed unscaled SGD

x = np.arange(len(methods))
width = 0.35

# RMSE comparison
axes[0].bar(x - width/2, rmse_lr_values, width, label='Linear Regression (OLS)', color='steelblue')
axes[0].bar(x + width/2, rmse_sgd_values, width, label='SGD Regressor (Normalized)', color='coral')
axes[0].set_title('1. RMSE Comparison')
axes[0].set_ylabel('RMSE')
axes[0].set_xticks(x, methods)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(bottom=10)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize='medium')


# Iterations to convergence
# Updated iterations list
iterations = [sgd_no_norm.n_iter_, sgd_std.n_iter_]
# Updated colors list
colors = ['#d62728', '#2ca02c']
bars = axes[1].bar(methods, iterations, color=colors)
axes[1].set_ylabel('Iterations')
axes[1].set_title('2. SGD Convergence Speed')
axes[1].grid(axis='y', alpha=0.3)
axes[1].bar_label(bars, padding=3, fontweight='bold')

# Coefficient comparison
# Updated DataFrame
coef_data = pd.DataFrame({
    'Unnormalized': lr_no_norm.coef_,
    'Standard (Z-score)': lr_std.coef_
}, index=feature_names)

# Sort by absolute coefficient for readability
coef_data = coef_data.reindex(coef_data['Standard (Z-score)'].abs().sort_values().index)

# Updated plot call
coef_data.plot(kind='barh', ax=axes[2],
               color={'Unnormalized':'lightgray', 'Standard (Z-score)':'blue'})
axes[2].set_xlabel('Coefficient Value (SymLog Scale)')
axes[2].set_title('3. Feature Importance (LR Coefficients)')
axes[2].grid(axis='x', alpha=0.3)
axes[2].set_xscale('symlog')
axes[2].legend(title='Normalization')

plt.tight_layout(rect=[0, 0.2, 1, 0.95])
plt.savefig('model_performance_summary.png', dpi=100, bbox_inches='tight')
plt.close()

In [14]:
# Beer NIR Dataset + PCA Regression Example

# Install Kaggle API
!pip install -q kaggle

# Upload your Kaggle API key (kaggle.json)
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
# (In Colab: click the folder icon on the left, then upload kaggle.json)

# Download and unzip the dataset
!kaggle datasets download -d robertoschimmenti/beer-nir -p /content
!unzip -o /content/beer-nir.zip -d /content

# Load dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Load
df = pd.read_csv("/content/beer.csv")
y = df.iloc[:,0].values
X = df.iloc[:,1:].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features (CRITICAL for PCA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# MODEL 1: Regression on REDUCED Features (PCA)
pca = PCA(n_components=10) # Using 10 components
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
total_variance_explained = np.sum(pca.explained_variance_ratio_) # Capture explained variance

model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

# Evaluate PCA model
rmse_pca = np.sqrt(mean_squared_error(y_test, y_pred_pca))
r2_pca = r2_score(y_test, y_pred_pca)

# MODEL 2: Regression on ORIGINAL Features
model_orig = LinearRegression()
model_orig.fit(X_train_scaled, y_train) # Using all 576 features
y_pred_orig = model_orig.predict(X_test_scaled)

# Evaluate Original model
rmse_orig = np.sqrt(mean_squared_error(y_test, y_pred_orig))
r2_orig = r2_score(y_test, y_pred_orig)

# Plot Predicted vs Actual
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Plot 1: PCA Model
ax1.scatter(y_test, y_pred_pca, c='blue', edgecolors='k', alpha=0.7)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax1.set_xlabel("True Original Extract")
ax1.set_ylabel("Predicted Original Extract")
ax1.set_title(f"PCA (10 components) + Regression\n"
              f"R² = {r2_pca:.3f} | RMSE = {rmse_pca:.3f}\n"
              f"Variance Explained: {total_variance_explained*100:.1f}%")
ax1.grid()

# Plot 2: Original Model
ax2.scatter(y_test, y_pred_orig, c='green', edgecolors='k', alpha=0.7)
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax2.set_xlabel("True Original Extract")
ax2.set_ylabel("Predicted Original Extract")
ax2.set_title(f"Original (576 features) + Regression\n"
              f"R² = {r2_orig:.3f} | RMSE = {rmse_orig:.3f}\n"
              f"Features Used: 100%")
ax2.grid()

plt.tight_layout()
plt.savefig('model_reduction_summary.png', dpi=100, bbox_inches='tight')
plt.close()

Dataset URL: https://www.kaggle.com/datasets/robertoschimmenti/beer-nir
License(s): unknown
beer-nir.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/beer-nir.zip
  inflating: /content/beer.csv       
